# Projekt 1

### Wczytanie bibliotek

In [ ]:
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier, VotingClassifier, StackingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.feature_selection import SelectFromModel, SequentialFeatureSelector, VarianceThreshold
from sklearn.pipeline import Pipeline
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### Wczytanie plików

In [8]:
X_train = pd.read_csv("https://raw.githubusercontent.com/kozaka93/2025Z-MachineLearning/refs/heads/main/project/artifical_train_data.csv")
y_train = pd.read_csv("https://raw.githubusercontent.com/kozaka93/2025Z-MachineLearning/refs/heads/main/project/artifical_train_labels.csv")
X_test = pd.read_csv("https://raw.githubusercontent.com/kozaka93/2025Z-MachineLearning/refs/heads/main/project/artifical_test_data.csv")

In [3]:
X_train

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V91,V92,V93,V94,V95,V96,V97,V98,V99,V100
0,500,441,400,445,519,447,479,484,549,424,...,436,477,496,458,454,526,563,448,518,454
1,535,454,513,437,502,471,506,490,371,441,...,519,484,500,460,493,497,489,462,489,500
2,473,469,527,487,478,461,454,438,419,532,...,537,492,467,480,449,474,481,464,463,523
3,490,480,518,453,487,468,454,473,533,536,...,518,487,485,457,460,465,445,468,511,470
4,443,472,504,511,481,487,497,469,416,559,...,511,472,483,502,481,514,513,483,474,530
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1495,511,485,601,516,522,449,479,469,305,500,...,588,484,523,483,478,486,464,493,472,496
1496,463,513,459,506,486,480,484,544,434,572,...,480,476,489,477,467,484,510,460,472,483
1497,504,450,441,501,476,435,484,501,582,485,...,459,475,468,440,481,487,423,468,445,491
1498,457,483,503,499,465,505,509,485,467,410,...,513,466,499,482,467,483,459,447,500,473


In [4]:
#Sprawdzam, czy dane zawierają braki
X_train.isna().any().any()

np.False_

In [5]:
#Sprawdzam typy danych
X_train.dtypes.value_counts()

int64    100
Name: count, dtype: int64

In [6]:
X_train.describe()

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V91,V92,V93,V94,V95,V96,V97,V98,V99,V100
count,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,...,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000
mean,478.896667,479.015333,487.754000,478.756667,482.687333,478.743333,481.036667,481.806000,476.974000,495.835333,...,499.702667,480.734667,491.862000,483.486667,483.852667,493.208000,484.626667,482.319333,478.442667,490.217333
std,26.423515,29.103558,73.378083,33.332599,13.858967,22.437790,22.502701,45.535481,103.426621,58.082872,...,52.959684,6.698919,19.349261,18.051049,16.290607,19.019268,36.665691,20.102938,28.380598,33.188297
min,379.000000,361.000000,282.000000,366.000000,435.000000,394.000000,409.000000,347.000000,180.000000,348.000000,...,351.000000,459.000000,425.000000,425.000000,420.000000,436.000000,354.000000,422.000000,391.000000,385.000000
25%,461.000000,460.000000,428.000000,456.000000,473.000000,464.000000,467.000000,450.000000,406.000000,447.000000,...,455.000000,476.000000,479.000000,471.000000,473.000000,481.000000,460.000000,468.000000,460.000000,467.000000
50%,478.000000,479.000000,490.000000,479.000000,483.000000,479.000000,481.000000,482.000000,477.000000,496.000000,...,501.000000,481.000000,492.000000,483.000000,484.000000,493.000000,484.000000,482.000000,479.000000,490.000000
75%,497.000000,498.000000,547.000000,501.000000,492.000000,494.000000,496.000000,513.000000,551.000000,542.250000,...,542.000000,485.000000,505.000000,496.000000,495.000000,506.000000,508.000000,495.000000,497.000000,513.000000
max,563.000000,571.000000,677.000000,593.000000,535.000000,553.000000,566.000000,621.000000,795.000000,651.000000,...,653.000000,507.000000,558.000000,549.000000,545.000000,561.000000,607.000000,551.000000,587.000000,608.000000


In [7]:
y_train

,Class
0,2
1,1
2,1
3,1
4,1
...,...
1495,1
1496,2
1497,1
1498,2


In [ ]:
y_train['Class'].value_counts(normalize=True)
#sprawdzam jaką część y stanowią

Class
1    0.503333
2    0.496667
Name: proportion, dtype: float64

### Tworzenie modeli

In [70]:
modele = []
BA = []

#### 1. Regresja logistyczna

In [ ]:
pipeline_rl = Pipeline([
    ('scaler', StandardScaler()),
    ('const', VarianceThreshold(threshold=0.05)),
    ('selector', SelectFromModel(estimator = LogisticRegression(penalty = 'l1', solver = 'saga', max_iter = 3000))),
    ('model', LogisticRegression())
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=333051)

param_grid_rl = {
    # selekcja cech, wybór selektora
    'selector': [
        SelectFromModel(
            estimator= RandomForestClassifier(random_state=333051)
        ),
        SelectFromModel(
            estimator=LogisticRegression(
                penalty='l1', solver='saga', max_iter=5000, random_state=333051
            )
        )
    ],

    #parametry regresji
    'model__C': [0.1, 1, 10]

}

grid1 = GridSearchCV(
    pipeline_rl,
    param_grid=param_grid_rl,
    cv=cv,
    scoring='balanced_accuracy',
    n_jobs=-1
)

In [ ]:
grid1.fit(X_train, y_train['Class'])
print(grid1.best_score_) #regresja uzyskała słaby wynik 

0.6107382550335572


In [71]:
modele.append("Regresja logistyczna")
BA.append(grid1.best_score_)

In [13]:
grid1.best_params_

{'model__C': 10,
 'selector': SelectFromModel(estimator=RandomForestClassifier(random_state=333051))}

### 2. KNN

In [5]:
pipeline_knn = Pipeline([
    ('scaler', StandardScaler()),
    ('const', VarianceThreshold(threshold=0.05)),
    ('selector', SelectFromModel(estimator = LogisticRegression(penalty = 'l1', solver = 'saga', max_iter = 3000))),
    ('model', KNeighborsClassifier())
])

param_grid_knn = {
    # selekcja cech, wybór selektora
    'selector': [
        SelectFromModel(
            estimator= RandomForestClassifier(random_state=333051)
        ),
        SelectFromModel(
            estimator=LogisticRegression(
                penalty='l1', solver='saga', max_iter=5000, random_state=333051
            )
        )
    ],

    #parametry knn
    'model__n_neighbors': [3, 5, 10, 15],
    'model__weights': ['uniform', 'distance']

}

grid2 = GridSearchCV(
    pipeline_knn,
    param_grid=param_grid_knn,
    cv=cv,
    scoring='balanced_accuracy',
    n_jobs=-1
)


In [14]:
grid2.fit(X_train, y_train.Class)
grid2.best_score_

np.float64(0.8826347837681675)

In [72]:
modele.append('KNN')
BA.append(grid2.best_score_)

In [18]:
grid2.best_params_

{'model__n_neighbors': 10,
 'model__weights': 'distance',
 'selector': SelectFromModel(estimator=RandomForestClassifier(random_state=333051))}

### 3. SVC

In [30]:
pipeline_svc = Pipeline([
    ('scaler', StandardScaler()),
    ('const', VarianceThreshold(threshold=0.05)),
    ('selector', SelectFromModel(estimator = LogisticRegression(penalty = 'l1', solver = 'saga', max_iter = 3000))),
    ('model', SVC(probability= True))
])

param_grid_svc = {
    # selekcja cech, wybór selektora
    'selector': [
        SelectFromModel(
            estimator= RandomForestClassifier(random_state=333051)
        ),
        SelectFromModel(
            estimator=LogisticRegression(
                penalty='l1', solver='saga', max_iter=5000, random_state=333051
            )
        )
    ],

    #parametry svc
    'model__C': [0.5, 1, 5, 10],
    'model__gamma': ['scale', 'auto']

}
grid3 = GridSearchCV(
    pipeline_svc,
    param_grid=param_grid_svc,
    cv=cv,
    scoring='balanced_accuracy',
    n_jobs=-1
)





In [31]:
grid3.fit(X_train, y_train.Class)

grid3.best_params_

{'model__C': 5,
 'model__gamma': 'scale',
 'selector': SelectFromModel(estimator=RandomForestClassifier(random_state=333051))}

In [32]:
print(grid3.best_score_)

0.8539268411929418


In [73]:
modele.append('SVC')
BA.append(grid3.best_score_)

### 4. Random Forest

In [11]:
pipeline_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('const', VarianceThreshold(threshold=0.05)),
    ('selector', SelectFromModel(estimator = LogisticRegression(penalty = 'l1', solver = 'saga', max_iter = 3000))),
    ('model', RandomForestClassifier())
])

param_grid_rf = {
    # selekcja cech, wybór selektora
    'selector': [
        SelectFromModel(
            estimator= RandomForestClassifier(random_state=333051)
        ),
        SelectFromModel(
            estimator=LogisticRegression(
                penalty='l1', solver='saga', max_iter=5000, random_state=333051
            )
        )
    ],

    #parametry rf
    'model__n_estimators': [100, 300],
    'model__max_depth': [None, 8, 15],
    'model__min_samples_leaf': [1, 5],
    'model__max_features': ['sqrt', 0.5]

}
grid4 = GridSearchCV(
    pipeline_rf,
    param_grid=param_grid_rf,
    cv=cv,
    scoring='balanced_accuracy',
    n_jobs=-1
)

In [12]:
grid4.fit(X_train, y_train['Class'])
print(grid4.best_score_)
print(grid4.best_params_)

0.8840659584870438
{'model__max_depth': 15, 'model__max_features': 0.5, 'model__min_samples_leaf': 1, 'model__n_estimators': 100, 'selector': SelectFromModel(estimator=RandomForestClassifier(random_state=333051))}


In [74]:
modele.append('RandomForest')
BA.append(grid4.best_score_)

## Stacking

In [ ]:
#bierzemy powyższe modele o najlepszych parametrach
model_1 = grid1.best_estimator_.named_steps['model']
model_2 = grid2.best_estimator_.named_steps['model']
model_3 = grid3.best_estimator_.named_steps['model']
model_4 = grid4.best_estimator_.named_steps['model']


#jako selector bierzemy RandomForrest, bo najlepiej sobie poradził w powyższych przypadkach

#1 przypadek- jako final_estimator bierzemy regresję logistyczną
pipeline_stacking1 = Pipeline([
    ('scaler', StandardScaler()),
    ('const', VarianceThreshold(threshold=0.05)),
    ('selector', SelectFromModel(RandomForestClassifier(random_state=333051))),
    ('model', StackingClassifier(estimators=[('lr', model_1),
                             ('rf', model_4),
                             ('knn', model_2),
                             ('svc', model_3)],
                             final_estimator=LogisticRegression(penalty='l1', solver='saga', max_iter=3000)))
])

param_grid_stacking1 = {

    #parametry selectora
    'selector__estimator__n_estimators': [100, 300],
    'selector__estimator__max_depth': [None, 8, 15],
    'selector__estimator__min_samples_leaf': [1, 5],
    
    # parametry final_estimator
    'model__final_estimator__C': [0.1, 1, 10]

}

grid_stacking1 = GridSearchCV(
    pipeline_stacking1,
    param_grid=param_grid_stacking1,
    cv=cv,
    scoring='balanced_accuracy',
    n_jobs=-1
)

In [20]:
grid_stacking1.fit(X_train, y_train.Class)
print(grid_stacking1.best_score_)

0.891986310502689


In [75]:
modele.append('Stacking z regresją')
BA.append(grid_stacking1.best_score_)

In [61]:
#2 przypadek- jako final_estoimator biorę Random forest
pipeline_stacking2 = Pipeline([
    ('scaler', StandardScaler()),
    ('const', VarianceThreshold(threshold=0.05)),
    ('selector', SelectFromModel(RandomForestClassifier(random_state=333051))),
    ('model', StackingClassifier(estimators=[('lr', model_1),
                             ('rf', model_4),
                             ('knn', model_2),
                             ('svc', model_3)],
                             final_estimator=RandomForestClassifier(random_state= 333051)))
])

param_grid_stacking2 = {

    #parametry selectora
    'selector__estimator__n_estimators': [100, 300],
    'selector__estimator__max_depth': [None, 8, 15],
    'selector__estimator__min_samples_leaf': [1, 5],
    
    # parametry final_estimator
    'model__final_estimator__n_estimators': [100, 300],
    'model__final_estimator__max_depth': [None, 8],
    'model__final_estimator__min_samples_leaf': [1, 5]
}

grid_stacking2 = GridSearchCV(
    pipeline_stacking2,
    param_grid=param_grid_stacking2,
    cv=cv,
    scoring='balanced_accuracy',
    n_jobs=-1
)


In [62]:
grid_stacking2.fit(X_train, y_train.Class)
grid_stacking2.best_score_

np.float64(0.8906351393395262)

In [76]:
modele.append('Stacking z lasem')
BA.append(grid_stacking2.best_score_)

### Voting

In [40]:
model_3 = grid3.best_estimator_.named_steps['model']
pipeline_voting = Pipeline([
    ('scaler', StandardScaler()),
    ('const', VarianceThreshold(threshold=0.05)),
    ('selector', SelectFromModel(RandomForestClassifier(random_state=333051, n_jobs=-1))),
    ('model', VotingClassifier(estimators=[('lr', model_1),
                             ('rf', model_4),
                             ('knn', model_2),
                             ('svc', model_3)], voting= 'soft'))
                            
])

param_voting = {
    
    #parametry selectora
    'selector__estimator__n_estimators': [100, 300],
    'selector__estimator__max_depth': [None, 8, 15],
    'selector__estimator__min_samples_leaf': [1, 5],
    
}

grid_voting = GridSearchCV(
    pipeline_voting,
    param_grid=param_voting,
    cv=cv,
    scoring='balanced_accuracy',
    n_jobs=-1
)

In [41]:
grid_voting.fit(X_train, y_train.Class)
grid_voting.best_score_

np.float64(0.8886839415085113)

In [77]:
modele.append('voting')
BA.append(grid_voting.best_score_)

In [ ]:
#tworzymy tabelę z modelami i wartościami metryki
df_wyniki = pd.DataFrame({
    'Model': modele,
    'Balanced Accuracy': BA
})

df_wyniki = df_wyniki.sort_values(
    by='Balanced Accuracy',
    ascending=False
).reset_index(drop=True)

df_wyniki


,Model,Balanced Accuracy
0,Stacking z regresją,0.891986
1,Stacking z lasem,0.890635
2,voting,0.888684
3,RandomForest,0.884066
4,KNN,0.882635
5,SVC,0.853927
6,Regresja logistyczna,0.610738


In [79]:
# wyciągamy najlepszy 
final_model = grid_stacking1.best_estimator_


# predykcja prawdopodobieństw
y_test_proba = final_model.predict_proba(X_test)[:, 0]


In [ ]:
#zapisujemy do pliku
np.savetxt(
    "333051_artifical_prediction.txt",
    y_test_proba,
    fmt="%.6f"
)